In [ ]:
from fidelity_option_data_downloader import FidelityOptionDataDownloader
from option_analyzer import *
self = OptionAnalyzer('quotes', 'chain')
ocd = FidelityOptionDataDownloader('chain', 'quotes', 'cookie.txt', logger=self.logger)
pd.set_option("display.max_columns", None)

def save_walls(walls, wall_file):
    with open(wall_file, 'w') as wfo:
        for s, w in walls.items():
            wfo.write(f'{s} {w}\n')

In [ ]:
os.system('sync > /dev/null 2>&1')
option_type = 'both'
servers = sorted(set([f.split('~')[1] for f in glob(os.path.expanduser(f'~/lab/data/{option_type}~*~*.csv'))]))
latest_option_files = [sorted(glob(os.path.expanduser(f'~/lab/data/{option_type}~{svr}~*.csv')))[-1] for svr in servers]
print('\n'.join(['%40s ' % os.path.basename(_) + ' '.join(self.list_symbols_in_data_file(_)) for _ in latest_option_files]))
chain_file_mtimes = dict([(os.path.basename(_f), os.path.getmtime(_f)) for _f in glob(os.path.expanduser('~/lab/chain/*'))])
latest_symbol = sorted(chain_file_mtimes, key=chain_file_mtimes.get)[-1]
print('Last symbol:', latest_symbol, datetime.fromtimestamp(chain_file_mtimes[latest_symbol]).strftime('%F %T'))

# Read option data processed by servers
dfcp = pd.concat([pd.read_csv(_f) for _f in latest_option_files])
dfcp = dfcp[~dfcp.symbol.str.contains('TLT|GLD')] # GEX is useless for TLT and GLD

In [ ]:
px.bar(self.calc_overall_put_call_ratios(dfcp), x='symbol', y=['OpenInterest_P_C_ratio', 'Volume_P_C_ratio'], barmode='group', width=2000, height=400).show()
total_net_gex, strike_gex = self.do_gex(dfcp)
self.plot_total_gex(total_net_gex, top_n=5, W=total_net_gex.shape[0]*80)

In [ ]:
__df = total_net_gex
__df = __df[__df.symbol.str.contains('SPY|QQQ|MRVL|LITE')]
put_walls, call_walls = self.plot_gex_profiles(strike_gex, __df, R=0.2, W=2000)
print('put_walls =', put_walls)
print('call_walls =', call_walls)
df_skew = self.calculate_iv_skew(dfcp[(dfcp.dte <= 45)].merge(__df.loc[:, ['symbol']], on='symbol', how='inner'))
px.bar(df_skew, x='expDt', y='25_delta_skew', color='symbol', barmode='group', height=500).show()
px.bar(df_skew, x='expDt', y='10_delta_skew', color='symbol', barmode='group', height=500).show()

In [ ]:
top_n = 25
__df = total_net_gex.sort_values(by='min_gex').head(top_n)
put_walls, call_walls = self.plot_gex_profiles(strike_gex, __df, R=0.1, W=2000)
print('put_walls =', put_walls)
save_walls(put_walls, 'put_walls.txt')
print('call_walls =', call_walls)
save_walls(call_walls, 'call_walls.txt')
df_skew = self.calculate_iv_skew(dfcp[(dfcp.dte <= 45)].merge(__df.loc[:, ['symbol']], on='symbol', how='inner'))
px.bar(df_skew, x='expDt', y='25_delta_skew', color='symbol', barmode='group', height=500).show()
px.bar(df_skew, x='expDt', y='10_delta_skew', color='symbol', barmode='group', height=500).show()

In [ ]:
__df = total_net_gex.sort_values(by='min_gex').iloc[15:]
put_walls, call_walls = self.plot_gex_profiles(strike_gex, __df, R=0.05, W=2000)
print('put_walls =', put_walls)
print('call_walls =', call_walls)
df_skew = self.calculate_iv_skew(dfcp[(dfcp.dte <= 45)].merge(__df.loc[:, ['symbol']], on='symbol', how='inner'))
px.bar(df_skew, x='expDt', y='25_delta_skew', color='symbol', barmode='group', height=500).show()
px.bar(df_skew, x='expDt', y='10_delta_skew', color='symbol', barmode='group', height=500).show()

In [ ]:
__df = total_net_gex.sort_values(by='min_gex').iloc[16:]
put_walls, call_walls = self.plot_gex_profiles(strike_gex, __df, R=0.05, W=2000)
print('put_walls =', put_walls)
print('call_walls =', call_walls)
df_skew = self.calculate_iv_skew(dfcp[(dfcp.dte <= 45)].merge(__df.loc[:, ['symbol']], on='symbol', how='inner'))
px.bar(df_skew, x='expDt', y='25_delta_skew', color='symbol', barmode='group', height=500).show()
px.bar(df_skew, x='expDt', y='10_delta_skew', color='symbol', barmode='group', height=500).show()

In [ ]:
__df = total_net_gex.sort_values(by='min_gex').iloc[20:]
put_walls, call_walls = self.plot_gex_profiles(strike_gex, __df, R=0.05, W=2000)
print('put_walls =', put_walls)
print('call_walls =', call_walls)
df_skew = self.calculate_iv_skew(dfcp[(dfcp.dte <= 45)].merge(__df.loc[:, ['symbol']], on='symbol', how='inner'))
px.bar(df_skew, x='expDt', y='25_delta_skew', color='symbol', barmode='group', height=500).show()
px.bar(df_skew, x='expDt', y='10_delta_skew', color='symbol', barmode='group', height=500).show()

In [ ]:
sym_filter_1 = r'INTC|WDC|MRVL|SNDK'
_df = total_net_gex
_df = _df[_df.symbol.str.contains(sym_filter_1)]
self.plot_gex_profiles(strike_gex, _df, R=0.3, W=2000)

In [ ]:
sym_filter_2 = 'QQQ|SPY|IWM|TLT|GLD|DIA'
_df = total_net_gex
_df = _df[_df.symbol.str.contains(sym_filter_2)]
self.plot_gex_profiles(strike_gex, _df, R=0.05, W=2000)

In [ ]:
_df = total_net_gex
_df = _df[~_df.symbol.str.contains(sym_filter_1 + '|' + sym_filter_2)]
#_df = _df[_df.symbol == 'IBIT']
self.plot_gex_profiles(strike_gex, _df, R=0.2, W=2000)